# 07 — Model Training

**Goals**
- Train logistic regression, decision tree, random forest
- Loop over each leave-one-cycle-out fold
- Also train full-data versions and save to `models/`


In [ ]:
import pandas as pd
import json
import joblib
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(PROCESSED_DIR / 'labeled_dataset.csv')
with open(PROCESSED_DIR / 'lolo_folds.json') as f:
    folds = json.load(f)

FEATURE_COLS = [
    'temperature_C', 'humidity_pct', 'soil_moisture_pct',
    'moisture_trend', 'light_lux', 'hour_of_day'
]
TARGET = 'urgency'

le = LabelEncoder()
df['urgency_enc'] = le.fit_transform(df[TARGET])
print('Label classes:', list(le.classes_))
joblib.dump(le, MODELS_DIR / 'label_encoder.joblib')


In [ ]:
# Train on each LOLO fold
# Note: outdoor cycles contain only the <24h class, so logistic
# cannot be trained on outdoor folds (needs ≥2 classes).

fold_models = {}
for fold in folds:
    fid = fold['fold_id']
    train_mask = df.cycle_id.isin(fold['train_cycles'])
    X_train = df.loc[train_mask, FEATURE_COLS]
    y_train = df.loc[train_mask, 'urgency_enc']
    n_classes = y_train.nunique()
    fold_models[fid] = {}

    for name in ['logistic', 'decision_tree', 'random_forest']:
        if name == 'logistic' and n_classes < 2:
            print(f'SKIP logistic on {fid}: only 1 class in train')
            continue
        if name == 'logistic':
            m = LogisticRegression(max_iter=1000, random_state=42)
        elif name == 'decision_tree':
            m = DecisionTreeClassifier(max_depth=4, random_state=42)
        else:
            m = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42)
        m.fit(X_train, y_train)
        fold_models[fid][name] = m
        print(f'Trained {name:15s} on {fid}  (n={len(X_train)}, classes={n_classes})')


In [ ]:
# Full-data models (all cycles)
for name in ['logistic', 'decision_tree', 'random_forest']:
    if name == 'logistic':
        m = LogisticRegression(max_iter=1000, random_state=42)
    elif name == 'decision_tree':
        m = DecisionTreeClassifier(max_depth=4, random_state=42)
    else:
        m = RandomForestClassifier(n_estimators=50, max_depth=4, random_state=42)
    m.fit(df[FEATURE_COLS], df['urgency_enc'])
    joblib.dump(m, MODELS_DIR / f'{name}_full.joblib')
    print(f'Saved full-data {name}')

joblib.dump(fold_models, MODELS_DIR / 'fold_models.joblib')
print('Saved fold_models.joblib')
